## content

In [2]:
import base64
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage  # 建议从 core 导入
from dotenv import load_dotenv

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        # model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_API_BASE)


def encode_image(img_path, img_type='jpeg'):
      """ 将一张本地图片转换成 Base64 编码的 Data URI 字符串 """
      with open(img_path, "rb") as img_file:
            # 2. 修正 return 语句的缩进，并去掉中间的错误换行
            return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode('utf-8')}"


# 图像路径
img_path = "1.jpg"

# 获取图像 base64 编码字符串
base64_image = encode_image(img_path)

# 3. 修正 HumanMessage 的传参格式
response = model.invoke(
      [
            HumanMessage(
                  content=[
                        {"type": "text", "text": "这张图里有什么？"},
                        {
                              "type": "image_url",
                              # 注意这里：必须包在 url 字段的字典里
                              "image_url": {"url": base64_image}
                        }
                  ]
            )
      ]
)

print(response.content)

这张图描绘了一位**动漫风格的少女**，主要特征如下：

*   **外貌**：她有着一头蓬松、略显凌乱的浅紫灰色长发，刘海盖住额头。她的左眼是蓝灰色的，半睁着，看起来有一种刚睡醒的慵懒感；右眼闭着，正在**用右手背揉眼睛**，脸颊上有淡淡的红晕。
*   **穿着**：她身穿一件宽松的**白色带领衬衫**（看起来像睡衣或家居服），领口微微敞开。
*   **表情与氛围**：微微下撇嘴，加上揉眼睛的动作和眼神，整体呈现出一种**困倦、迷糊或略带起床气**的可爱感觉。
*   **背景与细节**：背景是纯灰色，画面右侧有类似“揉眼”的细小特效线条，整体画风柔和，色调偏淡雅。
*   *（注：从发色、发型、瞳色和服装来看，角色特征非常接近《崩坏：星穹铁道》中的**符玄**，不过有些同人图会做一些色彩上的微调。）*


##  content_blocks
### 输入的格式化

In [3]:
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        # model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_API_BASE)


def encode_image(img_path):
      """ 将一张本地图片转换成 Base64 编码的 Data URI 字符串 , 方便在文本中嵌入图片数据 """

      with open(img_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode("utf-8")


# 图像路径
img_path = "1.jpg"

# 获取图像 base64 编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
      [
            # 此种格式可用
            # HumanMessage(
            # content=[
            # {'type': 'text', 'text': ' 这张图里有什么？ '},
            # {
            # 'type': 'image_url',
            # "image_url": base64_image,
            # }
            # ]
            # )
            # 推荐的统一写法
            HumanMessage(
                  content_blocks=[
                        {'type': 'text', 'text': ' 这张图里有什么？ '},
                        {
                              'type': 'image',
                              'base64': base64_image,
                              'mime_type': 'image/png',
                        }
                  ]
            )

      ]
)
print(response.content)

这是一张二次元动漫风格的插画，图片里包含以下内容：

*   **主要人物**：一个淡紫色长发的少女。她的头发看起来有些蓬松凌乱，刘海微微遮住额头。
*   **动作与神态**：她正用左手揉着左眼，右眼睁开，瞳孔是漂亮的冰蓝色。她的嘴巴微微抿起，表情看起来像是**刚睡醒、带着些许困倦或不情愿**的样子，脸颊上还有淡淡的红晕。
*   **穿着**：她穿着一件宽松的白色翻领衬衫（看起来像是睡衣或家居服）。
*   **背景与细节**：背景是纯灰色的。人物头部右侧有一个小小的白色闪光符号（✨），经常用来表现一种“迷糊”或“呆萌”的感觉。整体色调偏柔和、清冷。

整体给人一种非常慵懒、可爱的感觉。


## **举例 2：** Anthropic 模型

In [ ]:
from dotenv import load_dotenv

load_dotenv()

load_dotenv(override=True)

model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      extra_body={"thinking": {"type": "enabled"}},
)

response = model.invoke(" 你好，一句话回答 ")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)